# **RAG Fusion**
RAG-Fusion is an enhanced version of the traditional Retrieval-Augmented Generation (RAG) model. In RAG-Fusion, after receiving a query, the model first generates related sub-queries using a large language model. These sub-queries help find more relevant documents. Instead of simply sending the retrieved documents to the model, RAG-Fusion uses a technique called Reciprocal Rank Fusion (RRF) to score and reorder the documents based on their relevance. The best-ranked documents are then used to generate a more accurate response.

Research Paper: [RAG Fusion](https://arxiv.org/pdf/2402.03367)

## **Initial Setup**

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# load embedding model
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# load data
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader("../data/context.csv", encoding="utf-8")
documents = loader.load()

## **Vector Database (Chroma)**

In [ ]:
# split documents
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
documents = text_splitter.split_documents(documents)

## **Retriever**

## **Reciprocal Rank Fusion Chain**

In [ ]:
# create vectorstore
import chromadb
from langchain_chroma import Chroma

chroma_client = chromadb.EphemeralClient()
vectorstore = Chroma.from_documents(
    documents,
    embeddings,
    client=chroma_client,
    collection_name="fusion_rag",
)

In [ ]:
# create retriever
retriever = vectorstore.as_retriever()

In [ ]:
# create llm
from langchain_openai import ChatOpenAI
llm = ChatOpenAI()

In [ ]:
# create chain
from langchain_core.output_parsers import StrOutputParser
from langsmith import Client
client = Client()
prompt = client.pull_prompt("langchain-ai/rag-fusion-query-generation")

In [ ]:
# generate queries
generate_queries = (
    prompt | ChatOpenAI(temperature=0) | StrOutputParser() | (lambda x: x.split("\n"))
)

In [ ]:
# rerank results
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        # Assumes the docs are returned in sorted order of relevance
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            previous_score = fused_scores[doc_str]
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

In [ ]:
# create chain
chain = generate_queries | retriever.map() | reciprocal_rank_fusion

In [ ]:
# check input schema
chain.input_schema.schema()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


{'title': 'PromptInput',
 'type': 'object',
 'properties': {'original_query': {'title': 'Original Query',
   'type': 'string'}}}

In [ ]:
# rerank results
chain.invoke("what are points on a mortgage")

## **RAG Chain**

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context.
If you don't find the answer in the context, just say that you don't know.

Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

rag_fusion_chain = (
    {
        "context": chain,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_fusion_chain.invoke("what are points on a mortgage")

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


'Points on a mortgage are a form of pre-paid interest that borrowers can offer to pay a lender in order to reduce the interest rate on the loan, resulting in a lower monthly payment.'

## **Preparing Data for Evaluation**

In [ ]:
# create dataframe
from datasets import Dataset
import pandas as pd
dataset = Dataset.from_dict(data)
df = pd.DataFrame(dataset)
df

## **Evaluation with Ragas**

[Ragas](https://docs.ragas.io/)로 RAG 파이프라인을 평가합니다.

- **Faithfulness**: 답변이 검색된 컨텍스트에 근거하는지 (환각 탐지)
- **Answer Relevancy**: 답변이 질문에 얼마나 관련 있는지
- **Context Precision**: 관련 컨텍스트가 상위에 랭크되는지 (`ground_truth` 활용)

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision

# ragas 0.2.x: from_dict()는 list of dicts 형태 필요
# ground_truth 있으므로 ContextPrecision도 사용 가능
eval_data = [
    {
        "user_input": q,
        "response": r,
        "retrieved_contexts": c,
        "reference": gt,
    }
    for q, r, c, gt in zip(data["query"], data["response"], data["context"], data["ground_truth"])
]
ragas_dataset = EvaluationDataset.from_dict(eval_data)

In [ ]:
result = evaluate(
    dataset=ragas_dataset,
    metrics=[Faithfulness(), AnswerRelevancy(), ContextPrecision()],
)
print(result)

In [ ]:
result.to_pandas()